Building a RAG System Using Langchain

In [6]:
!pip install -q -r requirements.txt

In [7]:
from langchain_community.document_loaders import DirectoryLoader,UnstructuredFileLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
# from langchain_vectorstores import FAISS
# from langchain_chains import RetrievalQA
from langchain_chroma import Chroma

/tmp/ipykernel_12648/1189862129.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader,UnstructuredFileLoader


In [8]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [9]:
!pwd

/content


In [10]:
docs_dir_path = "/content/docs"
vector_db_path = "/content/vector_db"
collection_name  = "document_collection"

In [11]:
#load the embedding mode
embedding = HuggingFaceEmbeddings()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
# directory loader
loader = DirectoryLoader(
    path=docs_dir_path,
    glob="./*.pdf",
    loader_cls=UnstructuredFileLoader,
    # show_progress=True,
)

# single file can also be loaded with UnstructuredFileLoader

In [13]:
!apt-get install poppler-utils

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [14]:
documents = loader.load()

In [7]:
!pip list | grep fitz
!pip list | grep pymupdf

fitz                                     0.0.1.dev2


In [8]:
!pip uninstall -y fitz

Found existing installation: fitz 0.0.1.dev2
Uninstalling fitz-0.0.1.dev2:
  Successfully uninstalled fitz-0.0.1.dev2


In [9]:
!pip install -U pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 73.2 MB/s eta 0:00:00


In [1]:
import fitz

print(fitz.__doc__)

PyMuPDF 1.27.2.3: Python bindings for the MuPDF 1.27.2 library.
Python 3.12 running on linux (64-bit).



In [15]:
print(type(documents))

<class 'list'>


In [18]:
# initializing the text splitter
text_splitter = CharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=500,
)

In [19]:
# splitting the text into the smaller chunks
text_chunks = text_splitter.split_documents(documents)

In [20]:
len(text_chunks)

35

In [21]:
text_chunks[0]

Document(metadata={'source': '/content/docs/6. Evolution.pdf'}, page_content='12083CHO7\n\nCHAPTER 6\n\nEVOLUTION\n\n6.1 6.2\n\n6.3\n\n6.4\n\n6.5\n\n6.6\n\n6.7\n\n6.8\n\n6.9\n\nOrigin of Life\n\nEvolution of Life Forms - A\n\nTheory\n\nWhat are the Evidences for Evolution?\n\nWhat is Adaptive Radiation?\n\nBiological Evolution Mechanism of Evolution Hardy - Weinberg Principle\n\nA Brief Account of Evolution\n\nOrigin and Evolution of Man\n\nEvolutionary Biology is the study of history of life forms on earth. What exactly is evolution? To understand the changes in flora and fauna that have occurred over millions of years on earth, we must have an understanding of the context of origin of life, i.e., evolution of earth, of stars and indeed of the universe itself. What follows is the longest of all the construed and conjectured stories. This is the story of origin of life and evolution of life forms or biodiversity on planet earth in the context of evolution of earth and against the backg

In [22]:
vector_store = Chroma.from_documents(
    documents=text_chunks,
    embedding=embedding,
    persist_directory=vector_db_path,
    collection_name=collection_name,
)

this is the code ingestion part where performing the etl that is extract , transform and load the pipeline of ai
1. extract : Read the raw text out of your PDF files.
2. transform(chunk): Chop that massive text into smaller, bite-sized paragraphs (e.g., 1000 characters each) so the AI can digest them.
3. Embed (Hash): Pass those text chunks through an embedding model. This translates the English words into massive arrays of numbers (vectors) so the computer can calculate mathematical similarity between concepts.
4. load : Save those numerical vectors into a Vector Database (like ChromaDB). This becomes the "long-term memory" or filing cabinet for your RAG system.

now the rag retrieval part is there

In [23]:
!pip install langchain-classic

In [25]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_classic.chains import RetrievalQA


In [ ]:
import os
os.environ['GROQ_API_KEY'] = ""

In [28]:
vector_db_path = "/content/vector_db"
collection_name = "document_collection"

In [29]:
#load the embedding model

embedding = HuggingFaceEmbeddings()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [30]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.0
)

In [31]:
vector_store= Chroma(
    embedding_function=embedding,
    persist_directory=vector_db_path,
    collection_name=collection_name,
)

In [33]:
retriever = vector_store.as_retriever()

In [34]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
)

In [36]:
query = "What does the document say about the Adaptive Radiation"
response = qa_chain.invoke({"query":query})
print(response["result"])
print("-"*80)
for source in response["source_documents"]:
  print(source.metadata)

According to the document, Adaptive Radiation is the process of evolution of different species in a given geographical area starting from a point and literally radiating to other areas of geography (habitats). This process occurs when a single ancestral species evolves into multiple species that occupy different habitats within the same geographical area.

The document provides two examples of Adaptive Radiation:

1. Darwin's Finches: These small black birds evolved from a single ancestral species into multiple species with different beak shapes and sizes, enabling them to occupy different ecological niches.
2. Australian Marsupials: These marsupials evolved from a single ancestral stock into multiple species, each occupying different habitats within the Australian continent.

The document also mentions that when multiple Adaptive Radiations occur in an isolated geographical area, it can be referred to as Convergent Evolution.
-----------------------------------------------------------

In [37]:
response

{'query': 'What does the document say about the Adaptive Radiation',
 'result': "According to the document, Adaptive Radiation is the process of evolution of different species in a given geographical area starting from a point and literally radiating to other areas of geography (habitats). This process occurs when a single ancestral species evolves into multiple species that occupy different habitats within the same geographical area.\n\nThe document provides two examples of Adaptive Radiation:\n\n1. Darwin's Finches: These small black birds evolved from a single ancestral species into multiple species with different beak shapes and sizes, enabling them to occupy different ecological niches.\n2. Australian Marsupials: These marsupials evolved from a single ancestral stock into multiple species, each occupying different habitats within the Australian continent.\n\nThe document also mentions that when multiple Adaptive Radiations occur in an isolated geographical area, it can be referred

In [38]:
query = "What does the document say about Evolution and Ecosystem?"
response = qa_chain.invoke({"query":query})
print(response["result"])
print("-"*50)
for source in response["source_documents"]:
    print(source.metadata)

The document discusses the following points related to Evolution and Ecosystem:

**Evolution:**

1. The origin of life on earth is understood against the background of the origin of the universe, especially the earth.
2. Most scientists believe that chemical evolution, i.e., the formation of biomolecules, preceded the appearance of the first cellular forms of life.
3. The subsequent events regarding the first form of life are a conjectured story based on Darwinian ideas of organic evolution by natural selection.
4. Diversity of life forms on earth has been changing over millions of years.
5. Variations in a population result in variable fitness, and other phenomena like habitat fragmentation and genetic drift may accentuate these variations leading to the appearance of new species and hence evolution.
6. Homology is accounted for by the idea of branching descent.
7. Study of comparative anatomy, fossils, and comparative biochemistry provides evidence for evolution.
8. The story of evol